# 01 · Prepare data

**Research question:** with the number of training characters held fixed, does increasing context length from 32 to 128 to 256 lower test bits-per-character (BPC) for a 4-layer character transformer on TinyShakespeare, by more than the run-to-run variation across 5 seeds?

This notebook sets up the environment and loads, splits and encodes the data. Notebook 02 runs it with `%run`, so later notebooks get the same variables.

**Before running:** in Colab, use *Runtime → Change runtime type → T4 GPU*. The model and training code are adapted from Karpathy's nanoGPT (MIT licence).

## 1. Setup

In [ ]:
# Optional (Colab): keep the repo on Google Drive so files survive a disconnect.
# Clone it once into MyDrive first: !git clone <repo-url> /content/drive/MyDrive/CS6514-AIML
import os
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/CS6514-AIML/notebooks")

In [ ]:
import os, json, math, time, random, hashlib, platform, urllib.request, csv
from collections import Counter, defaultdict
from contextlib import nullcontext
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import yaml

# Notebooks run from the notebooks/ folder; all settings live in ../config.yaml.
PROJECT_DIR = os.path.abspath("..")
with open(os.path.join(PROJECT_DIR, "config.yaml")) as f:
    CFG = yaml.safe_load(f)

DATA_DIR = os.path.join(PROJECT_DIR, CFG["paths"]["data_dir"])
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, CFG["paths"]["checkpoint_dir"])   # model weights, one folder per run
RESULTS_DIR = os.path.join(PROJECT_DIR, CFG["paths"]["results_dir"])         # small JSON/CSV/PNG results
RUNS_DIR = os.path.join(RESULTS_DIR, "runs")                                  # per-run config and metrics
for d in (DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, RUNS_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"     # fp16 mixed precision on GPU (a T4 has no bf16)

# Set experiment.quick_test in config.yaml only to check the pipeline quickly (tiny runs).
# Quick-test rows are flagged in the results CSV; empty results/ and checkpoints/ before real runs.
QUICK_TEST = CFG["experiment"]["quick_test"]

MAX_CTX = CFG["experiment"]["max_ctx"]    # longest context in the study; evaluation targets start at this index in every split
SEEDS = CFG["experiment"]["seeds"]
CONTEXT_LENGTHS = CFG["experiment"]["context_lengths"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()

env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "cuda": torch.version.cuda,
}
with open(os.path.join(RESULTS_DIR, "environment.json"), "w") as f:
    json.dump(env, f, indent=2)
print(env)

## 2. Data loading and preprocessing

- Source: `input.txt` from Karpathy's char-rnn repository (TinyShakespeare, public-domain text).
- **Contiguous** 80/10/10 split, because a random split would leak neighbouring text between sets.
- Vocabulary built from the **training split only**. Any character in validation/test that never appears in training is mapped to an `<unk>` symbol and reported.

In [ ]:
DATA_URL = CFG["data"]["url"]
raw_path = os.path.join(DATA_DIR, "input.txt")
if not os.path.exists(raw_path):
    urllib.request.urlretrieve(DATA_URL, raw_path)

with open(raw_path, encoding="utf-8") as f:
    text = f.read()

n = len(text)
frac_train, frac_val, _ = CFG["data"]["split"]
n_train, n_val = int(frac_train * n), int(frac_val * n)
train_text = text[:n_train]
val_text = text[n_train:n_train + n_val]
test_text = text[n_train + n_val:]

UNK = "\x00"                                  # placeholder for characters unseen in training
train_chars = sorted(set(train_text))
assert UNK not in train_chars
itos = train_chars + [UNK]
stoi = {c: i for i, c in enumerate(itos)}
VOCAB_SIZE = len(itos)

unseen = {name: sorted(set(t) - set(train_chars)) for name, t in [("val", val_text), ("test", test_text)]}

def map_unseen(t):
    return "".join(c if c in stoi else UNK for c in t)

val_text = map_unseen(val_text)
test_text = map_unseen(test_text)

def encode(t):
    return torch.tensor([stoi[c] for c in t], dtype=torch.long)

train_data, val_data, test_data = encode(train_text), encode(val_text), encode(test_text)

data_meta = {
    "source_url": DATA_URL,
    "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
    "total_chars": n,
    "split_sizes": {"train": len(train_text), "val": len(val_text), "test": len(test_text)},
    "split": f"contiguous {CFG['data']['split']}",
    "vocab_size_including_unk": VOCAB_SIZE,
    "unseen_chars_mapped_to_unk": {k: [repr(c) for c in v] for k, v in unseen.items()},
    "eval_targets_start_index": MAX_CTX,
}
with open(os.path.join(RESULTS_DIR, "data_meta.json"), "w") as f:
    json.dump(data_meta, f, indent=2)
print(json.dumps(data_meta, indent=2))
print("\nStart of the training text:\n", train_text[:200])